> **Research record.** Cleaned copy of `800-research-users-locations/1003-place2vec-3periods.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Trains place2vec embeddings for three periods (pre-COVID, immediately after, one year after) and measures how close places are to the "work" context. Helpers are in `place2vec.py` (settings in `place2vec_config.json`).
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

All phases implemented in place2vec.py. Here is a summary of what was added:

 - Phase 0 -- create_dwells_pois_home_work() (line 68)
Runs a single BQ DDL that: flags home/work, joins Google 3rd places (20m priority), falls back to OSM/Foursquare (20m), drops unmatched dwells, attaches census_id via geohash.

- Phase 1 -- Export + Tokenization (line 782+)
export_visits_to_gcs() -- BQ EXPORT DATA that left-joins base dwells with the new table, bringing back unmatched dwells for the corpus.
GCSCorpus class -- restartable Dask-based iterator that streams Parquet from GCS, applies weekday/duration filters, segments sessions (3-hour gap rule), filters rare POIs (< 5 occurrences), and tokenizes with 4 schemes (0=Baseline, 1=Spatial, 2=Admin, 3=Temporal).
build_home_work_census_lookup() -- lightweight BQ query for (identifier, month) -> (home_census, work_census).
write_corpus_to_gcs() / GCSLineSentences -- write/read text corpora on GCS for reuse.

- Phase 2 -- Training + Alignment (line 1110+)
train_from_corpus() -- trains gensim Word2Vec from any restartable iterable (default: vector_size=100, window=5, min_count=5, sg=1, negative=15).
align_model_procrustes() -- aligns a target model onto a source model using scipy.linalg.orthogonal_procrustes with shared HOME_/WORK* anchors.
export_for_projector() -- export vectors + metadata TSV for t-SNE (TensorFlow Projector).

- Phase 3 -- Validation (line 1210+)
validate_stability() -- matching rate of top-5 neighbours within same category.
validate_remote_work_rank() -- rank of ground-truth POIs in WORK similarity list.

- Phase 4 -- Analysis (line 1306+)
extract_top_ngrams() (Q1), compute_workiness() (Q2), get_neighbours() (Q3), correlation_geo_func() (Q4).

#### Important: For the paper we use schemas 1 and 3

In [ ]:
import sys
import importlib

project_root = "../.."
if project_root not in sys.path:
    sys.path.append(project_root)

import place2vec

importlib.reload(
    place2vec
)  # pick up latest place2vec.py (e.g. top_third_place_ngrams_by_home)

# ═══════════════════════════════════════════════════════════════════════════════
# Configuration
# ═══════════════════════════════════════════════════════════════════════════════

GS_BUCKET = "phd_remote_work_data"  # <-- replace
GS_PREFIX = "place2vec"  # folder inside bucke                           # tokenization scheme (0–3)
PERIODS = place2vec.PERIODS  # pre_covid, post_immediate, post_1year

## Step 0 - Create the BQ dwells_pois_home_work table  (run once)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 0 : Recreate the BQ dwells_pois_home_work table  (run once)
# ═══════════════════════════════════════════════════════════════════════════════

place2vec.create_dwells_pois_home_work(
    google_poi_radius_m=20,
    osm_poi_radius_m=20,
)

## Step 1 - Export visits per period to GCS  (run once per period)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1 : Export visits per period to GCS  (run once per period)
# ═══════════════════════════════════════════════════════════════════════════════

for period_name, month_list in PERIODS.items():
    place2vec.export_visits_to_gcs(
        period_name=period_name,
        month_list=month_list,
        gs_bucket=GS_BUCKET,
        gs_prefix=GS_PREFIX,
        if_exists="replace",
    )

# ══

### Statistic for the paper

In [ ]:
#num of trajectories dont depend on the scheme
import gcsfs
import pandas as pd

fs = gcsfs.GCSFileSystem()

SCHEME = 3  # set to the scheme you trained, e.g. 0, 1, 2, 3

counts = []
for period_name in PERIODS:
    uri = f"gs://{GS_BUCKET}/{GS_PREFIX}/corpus_{period_name}_scheme{SCHEME}.txt"
    n_sentences = 0

    with fs.open(uri, "r") as f:
        for _ in f:
            n_sentences += 1

    counts.append({
        "period": period_name,
        "scheme": SCHEME,
        "sentences": n_sentences,
        "uri": uri,
    })

pd.DataFrame(counts)

In [ ]:
# takes 30 minutes to load
df_dist = place2vec.visit_distribution_by_period(
    gs_bucket=GS_BUCKET,
    gs_prefix=GS_PREFIX,
    periods=PERIODS,  # defaults to PERIODS
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

period_list = list(PERIODS.keys())
n_periods = len(period_list)

fig, axes = plt.subplots(1, n_periods, figsize=(6 * n_periods, 5), sharey=True)

if n_periods == 1:
    axes = [axes]

for i, period in enumerate(PERIODS):
    ax = axes[i]
    data = df_dist[
        (df_dist.period == period)
        & (df_dist.location_type == "THIRD_PLACE")
        & (df_dist.visit_count >= 10)
    ]["visit_count"]
    sns.kdeplot(
        data,
        ax=ax,
    )
    ax.set_title(
        f"Visit count distribution\nTHIRD_PLACE – {period.replace('_', ' ').title()}"
    )
    ax.set_xlabel("Visit count")
    ax.set_ylabel("Location count")
    ax.grid(
        True,
        linestyle=":",
    )

plt.tight_layout()
plt.show()

## Step 2 - Build corpus, train Word2Vec, save  (per period)

Here's what each parameter does in `train_from_corpus`:

- **`corpus=line_corpus`** — The training data: an iterable of sentences, where each sentence is a list of tokens (e.g. `["HOME_1234", "POI_abc", "GH_sv8qf3"]`). Here it's a `GCSLineSentences` object reading from a GCS text file.

- **`vector_size=100`** — Dimensionality of the embedding vectors ( the size of hidden layer). Each token gets a 100-dimensional vector. Higher values capture more nuance but need more data and compute. 100 is a common starting point.

- **`window=5`** — Context window size. When training on a sentence, the model considers tokens up to 5 positions before and after the target token as context. Larger windows capture broader spatial/temporal co-occurrence; smaller windows capture tighter local relationships.

- **`min_count=5`** — Minimum frequency threshold. Tokens appearing fewer than 5 times across all sentences are discarded from the vocabulary. This filters out noise from extremely rare locations.

- **`sg=1`** — Training algorithm. `1` = Skip-gram (predicts context tokens from the target token). `0` would be CBOW (predicts target from context). Skip-gram typically works better for infrequent tokens, which is relevant here since many POIs may have low frequency.

- **`negative=15`** — Number of negative samples per positive example. During training, for each real (target, context) pair, the model draws 15 random "noise" tokens as negative examples. Higher values give more stable training for small datasets; 5–20 is typical.

- **`epochs=10`** — Number of full passes over the corpus. The model iterates through all sentences 10 times. More epochs generally improve quality but with diminishing returns and longer training time.

Here are the tokenization schemes implemented in `_tokenize_row` (0–5; schemes 0–3 are the originals, 4 is neighborhood + time, 5 is TLV + third-place/office):

### Scheme 0 — Baseline (Category-level)
- **HOME**: `HOME`
- **WORK**: `WORK`
- **POI**: `CAT_{category}` (e.g. `CAT_cafe`, `CAT_restaurant`)
- **Unknown**: `GH_{geohash6}` (6-char geohash, ~1.2km cell)

The coarsest scheme. All homes are the same token, all works are the same token, and POIs are merged by category. Good for discovering broad functional patterns (e.g. "cafes tend to appear near work").

### Scheme 1 — Spatial (Individual places)
- **HOME**: `HOME_{census_id}` (e.g. `HOME_1234567`)
- **WORK**: `WORK_{census_id}`
- **POI**: `POI_{poi_id}` (each POI is unique)
- **Unknown**: `GH_{geohash8}` (8-char geohash, ~40m cell)

Each place gets its own identity. Home/work are distinguished by their statistical area. The model learns place-specific embeddings — useful for finding which specific POIs are functionally similar.

### Scheme 2 — Administrative Flow
- **HOME**: `HOME_{census_id}`
- **WORK**: `WORK_{census_id}`
- **POI**: `POI_{poi_id}`
- **Unknown**: `CENSUS_{census_id}` (statistical area instead of geohash)

Same as Scheme 1 for matched POIs, but unmatched dwells are mapped to their statistical area rather than a raw geohash. This means the model can learn administrative-level co-occurrence even for places with no POI match — useful for understanding flows between neighborhoods.

### Scheme 3 — Temporal Context
- **HOME**: `HOME_{census_id}`
- **WORK**: `WORK_{census_id}`
- **POI**: `POI_{poi_id}_WEEKDAY_{period}` (e.g. `POI_abc_WEEKDAY_MORNING`)
- **Unknown**: `GH_{geohash8}_{period}` (e.g. `GH_sv8qf3xk_AFTERNOON`)

Adds a time-of-day suffix using 5 buckets: `MORNING` (6–9), `LATE_MORNING` (9–12), `AFTERNOON` (12–17), `EVENING` (17–23), `NIGHT` (23–6). The same cafe visited in the morning and evening becomes two different tokens, so the model can capture temporal usage patterns (e.g. "cafe-in-the-morning" is closer to "work" than "cafe-in-the-evening").
# 
### Scheme 4 — Neighborhood & Temporal Context
- **HOME**: `HOME_{home_neighborhood_type}`  
  Uses the home_census to look up the "new_neighborhood_type", resulting in tokens like `HOME_residential_central`, `HOME_satellite_city`, etc.
- **WORK**: `WORK_{work_neighborhood_type}`  
  Uses the work_census to look up the "new_neighborhood_type", giving tokens like `WORK_CBD`, `WORK_residential_remote`, etc.
- **POI**: `POI_{category}_{neighborhood_type}_WEEKDAY_{period}`  
  Each POI token encodes its category, its neighborhood type (from the POI's census), and the time-of-day bucket. Example: `POI_cafe_CBD_WEEKDAY_MORNING`.
- **Unknown/No POI (GH)**: `GH_{neighborhood_type}_WEEKDAY_{period}`  
  For unmatched POIs or generic locations, uses the filled census lookup to obtain the "new_neighborhood_type" and time period. Example: `GH_satellite_city_WEEKDAY_EVENING`.

    - `period` is one of: `MORNING`, `LATE_MORNING`, `AFTERNOON`, `EVENING`, `NIGHT`
    - If the neighborhood type is missing/unknown, use `"UNKNOWN"` in the token.

### Scheme 5 — TLV + temporal home + third-place / office_loc
Like Scheme 4, but replaces neighborhood type with a Tel Aviv flag, adds daytime to HOME, and tags POIs by location suitability for remote work.

- **HOME**: `HOME_{TLV|NOT_TLV}_WEEKDAY_{period}`  
  Home census id starting with `5000` → `TLV`, else `NOT_TLV`, plus the same day-part buckets as POI.
- **WORK**: `WORK_{TLV|NOT_TLV}`  
  Same TLV rule from the work census (no daytime suffix).
- **POI**: `POI_{category}_{TLV|NOT_TLV}_{third_place|office_loc}_WEEKDAY_{period}`  
  TLV from the dwell’s filled census; `office_loc` when `flag_location_suits_rw=0` on `phd_pois.google_pois_features`, otherwise `third_place` (including missing / OSM).
- **Unknown/No POI (GH)**: `GH_{TLV|NOT_TLV}_WEEKDAY_{period}`

Examples: `HOME_TLV_WEEKDAY_MORNING`, `POI_cafe_NOT_TLV_office_loc_WEEKDAY_AFTERNOON`.


### BUILD MODELS

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# STEP 2 : Build corpus, train Word2Vec, save  (per period)
# ═══════════════════════════════════════════════════════════════════════════════

models = {}
from tqdm import tqdm

for SCHEME in tqdm(range(4)):
    for period_name, month_list in PERIODS.items():

        # 2a. Build home/work census lookup (lightweight BQ query)
        hw_lookup = place2vec.build_home_work_census_lookup(month_list)

        # 2b. Create streaming corpus from GCS Parquet shards
        gs_parquet_uri = f"gs://{GS_BUCKET}/{GS_PREFIX}/visits_{period_name}/*.parquet"
        corpus = place2vec.GCSCorpus(
            gs_uri=gs_parquet_uri,
            scheme=SCHEME,
            home_work_lookup=hw_lookup,
            min_duration_hours=10 / 60,  # 10 minutes
            session_gap_hours=3,
            min_poi_count=10,
        )

        # 2c. (Optional) Materialise corpus to GCS text for reuse / inspection
        corpus_text_uri = (
            f"gs://{GS_BUCKET}/{GS_PREFIX}/corpus_{period_name}_scheme{SCHEME}.txt"
        )
        place2vec.write_corpus_to_gcs(corpus, corpus_text_uri, if_exists="replace")

        # 2d. Train Word2Vec from the materialised text corpus (restartable)
        line_corpus = place2vec.GCSLineSentences(corpus_text_uri)
        model = place2vec.train_from_corpus(
            corpus=line_corpus,
            vector_size=100,
            window=2,
            min_count=5,
            sg=1,
            negative=15,
            epochs=10,
        )

        # 2e. Save model locally
        model.save(f"models/place2vec_{period_name}_scheme{SCHEME}.model")
        models[f"{period_name}_scheme{SCHEME}"] = model
        print(f"✓ {period_name}: vocab={len(model.wv)}")

In [ ]:
from tqdm import tqdm

SCHEME = 4
for period_name, month_list in PERIODS.items():

    # 2a. Build home/work census lookup (lightweight BQ query)
    hw_lookup = place2vec.build_home_work_census_lookup(month_list)

    # 2b. Create streaming corpus from GCS Parquet shards
    gs_parquet_uri = f"gs://{GS_BUCKET}/{GS_PREFIX}/visits_{period_name}/*.parquet"
    corpus = place2vec.GCSCorpus(
        gs_uri=gs_parquet_uri,
        scheme=SCHEME,
        home_work_lookup=hw_lookup,
        min_duration_hours=10 / 60,  # 10 minutes
        session_gap_hours=3,
        min_poi_count=10,
    )

    # 2c. (Optional) Materialise corpus to GCS text for reuse / inspection
    corpus_text_uri = (
        f"gs://{GS_BUCKET}/{GS_PREFIX}/corpus_{period_name}_scheme{SCHEME}.txt"
    )
    place2vec.write_corpus_to_gcs(corpus, corpus_text_uri, if_exists="replace")

    # 2d. Train Word2Vec from the materialised text corpus (restartable)
    line_corpus = place2vec.GCSLineSentences(corpus_text_uri)
    model = place2vec.train_from_corpus(
        corpus=line_corpus,
        vector_size=100,
        window=2,
        min_count=5,
        sg=1,
        negative=15,
        epochs=10,
    )

    # 2e. Save model locally
    model.save(f"models/place2vec_{period_name}_scheme{SCHEME}.model")
    models[f"{period_name}_scheme{SCHEME}"] = model
    print(f"✓ {period_name}: vocab={len(model.wv)}")

### Scheme 5 — train (same hyperparams as Scheme 4)


In [ ]:
from tqdm import tqdm

SCHEME = 5
for period_name, month_list in PERIODS.items():

    # 2a. Build home/work census lookup (lightweight BQ query)
    hw_lookup = place2vec.build_home_work_census_lookup(month_list)

    # 2b. Create streaming corpus from GCS Parquet shards
    gs_parquet_uri = f"gs://{GS_BUCKET}/{GS_PREFIX}/visits_{period_name}/*.parquet"
    corpus = place2vec.GCSCorpus(
        gs_uri=gs_parquet_uri,
        scheme=SCHEME,
        home_work_lookup=hw_lookup,
        min_duration_hours=10 / 60,  # 10 minutes
        session_gap_hours=3,
        min_poi_count=10,
    )

    # 2c. (Optional) Materialise corpus to GCS text for reuse / inspection
    corpus_text_uri = (
        f"gs://{GS_BUCKET}/{GS_PREFIX}/corpus_{period_name}_scheme{SCHEME}.txt"
    )
    place2vec.write_corpus_to_gcs(corpus, corpus_text_uri, if_exists="replace")

    # 2d. Train Word2Vec from the materialised text corpus (restartable)
    line_corpus = place2vec.GCSLineSentences(corpus_text_uri)
    model = place2vec.train_from_corpus(
        corpus=line_corpus,
        vector_size=100,
        window=2,
        min_count=5,
        sg=1,
        negative=15,
        epochs=10,
        workers=1,
    )

    # 2e. Save model locally
    model.save(f"models/place2vec_{period_name}_scheme{SCHEME}.model")
    models[f"{period_name}_scheme{SCHEME}"] = model
    print(f"✓ {period_name}: vocab={len(model.wv)}")


In [ ]:
for name, model in models.items():
    # Single model
    print(name)
    vocab_size = len(model.wv.key_to_index)
    print("Vocabulary size:", vocab_size)
    print("----")

In [ ]:
for name, model in models.items():
    dist=[
         model.wv.get_vecattr(w, "count")
        for w in model.wv.key_to_index
]
    print (name,np.mean(dist))

# Load models (start here)

In [ ]:
# Load models
from tqdm import tqdm

models = dict()
from gensim.models import KeyedVectors

for SCHEME in tqdm(range(6)):
    for period_name, month_list in PERIODS.items():
        model = KeyedVectors.load(
            f"models/place2vec_{period_name}_scheme{SCHEME}.model", mmap="r"
        )
        models[f"{period_name}_scheme{SCHEME}"] = model


### Export for t-SNE (TensorFlow Projector)

Use `place2vec.export_for_projector()` to save vectors and metadata for [TensorFlow Projector](https://projector.tensorflow.org/) (t-SNE / PCA visualization). Run after models are loaded. With `category_map=None` (default), all tokens are exported with auto-derived categories (Geohash, Function, TimeOfDay, etc.). Set `include_geohashes=False` to export only attribute/concept tokens.

In [ ]:
# Export one model (e.g. pre_covid, scheme 0) for t-SNE
for SCHEME in [0, 4, 5]:
    for period_name in PERIODS:
        model = models[f"{period_name}_scheme{SCHEME}"]
        place2vec.export_for_projector(
            model,
            output_dir="projector_files",
            metadata_file=f"metadata_{period_name}_scheme{SCHEME}.tsv",
            vector_file=f"vectors_{period_name}_scheme{SCHEME}.tsv",
            category_map=None,  # auto-derived labels; or pass a dict token -> category
            include_geohashes=True,  # set False to exclude GH_* (fewer points, faster t-SNE)
        )


# Upload the two TSV files to https://projector.tensorflow.org/


### Scheme 4

In [ ]:
import place2vec

for neighborhood_type in [
    "CBD",
    "residential_central",
    "satellite_city",
    "residential_remote",
]:
    print(neighborhood_type)
    for SCHEME in [4]:
        for period_name in PERIODS:
            print(period_name)
            model = models[f"{period_name}_scheme{SCHEME}"]
            pairs = place2vec.closest_tokens(
                model, f"WORK_{neighborhood_type}", top_k=8, show=True
            )

In [ ]:
import place2vec


for SCHEME in [0]:
    for period_name in PERIODS:
        print(period_name)
        model = models[f"{period_name}_scheme{SCHEME}"]
        pairs = place2vec.closest_tokens(
            model, f"CAT_library", top_k=12, show=True
        )

### Choose anchors

### Anchor selection

- Restrict anchors to POIs that:

- Exist in all periods (same token in both models).

- Have enough visits in each period (above period‑specific thresholds).

- Don’t explode or vanish (e.g., max(visits)/min(visits) <= 3–5 across your three periods

In [ ]:
import numpy as np


def select_anchor_pois(
    df_dist,
    periods,
    keywords=None,
    min_visits_per_period=0,
    max_visit_ratio=3.0,
):
    """
    Select anchor POIs (POI_{id}) from visit_distribution_by_period output.

    Rules:
      - Exclude HOME / WORK; keep only location_id that look like POI ids.
      - If categories is provided, restrict to those pois_category values.
      - Exists in ALL given periods with visits >= period-specific thresholds.
      - Stable over time: max(visits) / min(visits) <= max_visit_ratio.
    """
    periods = list(periods)

    # Normalise thresholds: allow scalar or dict
    if isinstance(min_visits_per_period, (int, float)):
        min_vis = {p: min_visits_per_period for p in periods}
    else:
        min_vis = dict(min_visits_per_period)

    # 1) Base filter: drop HOME/WORK, keep chosen periods
    df = df_dist[
        (df_dist["period"].isin(periods))
        & (df_dist["location_id"] != "HOME")
        & (df_dist["location_id"] != "WORK")
    ].copy()

    # Optional category filter
    if keywords:
        keyword_lower = {c.lower() for c in keywords}
        df = df[df["keyword"].str.lower().isin(keyword_lower)]

    if df.empty:
        return []

    # 2) Pivot → visits per POI per period
    pivot = df.pivot_table(
        index="location_id",
        columns="period",
        values="visit_count",
        aggfunc="sum",
        fill_value=0,
    ).reindex(columns=periods, fill_value=0)
    if pivot.empty:
        return []

    # 3) Keep POIs above per-period thresholds
    mask = np.ones(len(pivot), dtype=bool)
    for p in periods:
        thr = min_vis.get(p, 0)
        mask &= pivot[p] >= thr
    filtered = pivot[mask]
    if filtered.empty:
        return []

    # 4) Stability filter: max/min <= max_visit_ratio
    visits = filtered.to_numpy()
    min_visits = visits.min(axis=1)
    max_visits = visits.max(axis=1)
    ratio = max_visits / np.maximum(min_visits, 1)
    stable_mask = ratio <= max_visit_ratio

    stable_ids = filtered.index[stable_mask]

    # 5) Return POI_{id} tokens
    return [f"POI_{pid}" for pid in stable_ids]

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_distances


def get_anchor_matrix(model, anchor_tokens):
    """
    model: gensim Word2Vec
    anchor_tokens: list of token strings, e.g. ["POI_123", "POI_456", ...]
    """
    wv = model.wv
    used = [t for t in anchor_tokens if t in wv.key_to_index]
    if not used:
        raise ValueError("No anchor tokens found in model vocabulary.")
    M = np.vstack([wv[t] for t in used])
    return M, used


def row_normalize(M):
    """L2-normalize each row."""
    norms = np.linalg.norm(M, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return M / norms


def common_anchors_across_models(models, anchor_tokens):
    """
    models: list of gensim Word2Vec models
    anchor_tokens: candidate tokens (e.g. POI_{id})
    """
    vocabs = [set(m.wv.key_to_index) for m in models]
    common = [t for t in anchor_tokens if all(t in vocab for vocab in vocabs)]
    return common


def pairwise_cosine_distance(M):
    """k x d → k x k pairwise cosine distance matrix."""
    return cosine_distances(M, M)


def upper_triangular_vector(D):
    """Flatten upper triangle (without diagonal) into a 1D vector."""
    k = D.shape[0]
    iu = np.triu_indices(k, k=1)
    return D[iu]

In [ ]:
def corr_anchor_alignment(SCHEME, aligned_models, anchors_list):
    model_pre = aligned_models[f"pre_covid_scheme{SCHEME}"]
    model_mid = aligned_models[f"post_immediate_scheme{SCHEME}"]
    model_post = aligned_models[f"post_1year_scheme{SCHEME}"]

    models_triplet = [model_pre, model_mid, model_post]

    # anchors_list is your POI_{id} list from select_anchor_pois
    shared_anchors = common_anchors_across_models(models_triplet, anchors_list)
    print("Shared anchors across all 3 models:", len(shared_anchors))

    A_pre, _ = get_anchor_matrix(model_pre, shared_anchors)
    A_mid, _ = get_anchor_matrix(model_mid, shared_anchors)
    A_post, _ = get_anchor_matrix(model_post, shared_anchors)

    # (optional) normalize rows
    # A_pre = row_normalize(A_pre); A_mid = row_normalize(A_mid); A_post = row_normalize(A_post)

    D_pre = pairwise_cosine_distance(A_pre)
    D_mid = pairwise_cosine_distance(A_mid)
    D_post = pairwise_cosine_distance(A_post)

    v_pre = upper_triangular_vector(D_pre)
    v_mid = upper_triangular_vector(D_mid)
    v_post = upper_triangular_vector(D_post)

    corr_pre_mid = np.corrcoef(v_pre, v_mid)[0, 1]
    corr_pre_post = np.corrcoef(v_pre, v_post)[0, 1]
    mad_pre_mid = np.mean(np.abs(v_pre - v_mid))
    mad_pre_post = np.mean(np.abs(v_pre - v_post))

    print("Correlation(pre, mid):", corr_pre_mid, "  MAD:", mad_pre_mid)
    print("Correlation(pre, post):", corr_pre_post, "  MAD:", mad_pre_post)
    return corr_pre_mid, mad_pre_mid, corr_pre_post, mad_pre_post

In [ ]:
keywords_candidates = ["religion", "school", "health", "playground", "pitch"]

In [ ]:
from itertools import combinations

all_combos = [
    combo
    for k in range(1, len(keywords_candidates) - 1)
    for combo in combinations(keywords_candidates, k)
]

### Alignment for Scheme 1

In [ ]:
SCHEME = 1
min_anchor = 10
dict_corr_mad = {}
for keywords in all_combos:
    for min_visits in [20, 30, 40]:
        for max_visit_ratio in [1.3, 1.4, 1.5, 1.6]:
            anchors_list = select_anchor_pois(
                df_dist,
                PERIODS,
                keywords=keywords,
                min_visits_per_period=min_visits,
                max_visit_ratio=max_visit_ratio,
            )
            if len(anchors_list) > min_anchor:
                model_ref = models[f"pre_covid_scheme{SCHEME}"]

                aligned_models = {f"pre_covid_scheme{SCHEME}": model_ref}

                for period_name in ["post_immediate", "post_1year"]:
                    aligned, R = place2vec.align_model_procrustes(
                        model_source=model_ref,
                        model_target=models[f"{period_name}_scheme{SCHEME}"],
                        anchor_tokens=anchors_list,  # auto-detect HOME_*/WORK_*
                        min_anchor=min_anchor,
                        scheme=SCHEME,
                    )

                    aligned_models[f"{period_name}_scheme{SCHEME}"] = aligned
                dict_corr_mad[f"{keywords}_{min_visits}_{max_visit_ratio}"] = (
                    corr_anchor_alignment(SCHEME, aligned_models, anchors_list)
                )

In [ ]:
import numpy as np

best_key = None
best_score = -np.inf

best_key_c1 = None
best_score_c1 = -np.inf
best_key_c2 = None
best_score_c2 = -np.inf

for key, stats in dict_corr_mad.items():
    c1 = stats[0]
    c2 = stats[2]

    if c1 > best_score_c1:
        best_score_c1 = c1
        best_key_c1 = key

    if c2 > best_score_c2:
        best_score_c2 = c2
        best_key_c2 = key

print("Best key for c1:", best_key_c1)
print("Best c1:", best_score_c1)
keywords_part_c1 = best_key_c1.rsplit("_", 2)[0]
print("Best keywords for c1:", keywords_part_c1)

print("Best key for c2:", best_key_c2)
print("Best c2:", best_score_c2)
keywords_part_c2 = best_key_c2.rsplit("_", 2)[0]

### Alignment for Scheme 3

In [ ]:
SCHEME = 3
min_anchor = 10
dict_corr_mad = {}
for keywords in all_combos:
    for min_visits in [20, 30, 40]:
        for max_visit_ratio in [1.3, 1.4, 1.5, 1.6]:
            anchors_list = select_anchor_pois(
                df_dist,
                PERIODS,
                keywords=keywords,
                min_visits_per_period=min_visits,
                max_visit_ratio=max_visit_ratio,
            )
            if len(anchors_list) > min_anchor:
                model_ref = models[f"pre_covid_scheme{SCHEME}"]

                aligned_models = {f"pre_covid_scheme{SCHEME}": model_ref}

                for period_name in ["post_immediate", "post_1year"]:
                    aligned, R = place2vec.align_model_procrustes(
                        model_source=model_ref,
                        model_target=models[f"{period_name}_scheme{SCHEME}"],
                        anchor_tokens=anchors_list,  # auto-detect HOME_*/WORK_*
                        min_anchor=min_anchor,
                        scheme=SCHEME,
                    )

                    aligned_models[f"{period_name}_scheme{SCHEME}"] = aligned
                dict_corr_mad[f"{keywords}_{min_visits}_{max_visit_ratio}"] = (
                    corr_anchor_alignment(SCHEME, aligned_models, anchors_list)
                )

In [ ]:
keywords_candidates = ["religion", "school", "mall", "health", "playground", "pitch"]

In [ ]:
from itertools import combinations

all_combos = [
    combo
    for k in range(1, len(keywords_candidates) - 1)
    for day_period in [
        "WEEKDAY_MORNING",
        "WEEKDAY_AFTERNOON",
        "WEEKDAY_EVENING",
        "WEEKEND_NIGHT",
        "WEEKEND_LATE_MORNING",
    ]
    for combo in combinations(keywords_candidates, day_period, k)
]

In [ ]:
min_anchor = 10
dict_corr_mad = {}
SCHEME = 4
# Scheme 4 has no POI_{id} tokens; use shared HOME_*/WORK_* across models
models_triplet = [
    models["pre_covid_scheme4"],
    models["post_immediate_scheme4"],
    models["post_1year_scheme4"],
]
for keywords in all_combos:
    for min_visits in [20, 30, 40]:
        for max_visit_ratio in [1.3, 1.4, 1.5, 1.6]:
            anchors_list = place2vec.get_anchor_tokens_home_work_poi(models_triplet)
            if len(anchors_list) < min_anchor:
                print(
                    f"Scheme 4: only {len(anchors_list)} HOME_*/WORK_* anchors (min_anchor={min_anchor}). Lower min_anchor or skip."
                )
            else:
                model_ref = models["pre_covid_scheme4"]
                aligned_models = {"pre_covid_scheme4": model_ref}
                for period_name in ["post_immediate", "post_1year"]:
                    aligned, R = place2vec.align_model_procrustes(
                        model_source=model_ref,
                        model_target=models[f"{period_name}_scheme4"],
                        anchor_tokens=anchors_list,
                        min_anchor=min_anchor,
                        scheme=4,
                    )
                    aligned_models[f"{period_name}_scheme4"] = aligned
                dict_corr_mad[f"{keywords}_{min_visits}_{max_visit_ratio}_scheme4"] = (
                    corr_anchor_alignment(SCHEME, aligned_models, anchors_list)
                )

#### Visualize results

In [ ]:
import pandas as pd

rows = []
for key, stats in dict_corr_mad.items():
    # Split key: everything before last 2 "_" is keywords_str
    keywords_str, min_vis_str, max_ratio_str = key.rsplit("_", 2)
    rows.append(
        {
            "key": key,
            "keywords_str": keywords_str,
            "n_keywords": keywords_str.count(",") + 1 if "," in keywords_str else 1,
            "min_visits": int(min_vis_str),
            "max_visit_ratio": float(max_ratio_str),
            "corr_pre_mid": stats[0],
            "corr_pre_post": stats[2],
        }
    )

df_corr = pd.DataFrame(rows)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

sc1 = axes[0].scatter(
    df_corr["min_visits"],
    df_corr["corr_pre_mid"],
    c=df_corr["max_visit_ratio"],
    s=20 + 10 * df_corr["n_keywords"],
    cmap="viridis",
)
axes[0].set_title("corr_pre_mid vs min_visits")
axes[0].set_xlabel("min_visits")
axes[0].set_ylabel("correlation")
cbar1 = plt.colorbar(sc1, ax=axes[0])
cbar1.set_label("max_visit_ratio")

sc2 = axes[1].scatter(
    df_corr["min_visits"],
    df_corr["corr_pre_post"],
    c=df_corr["max_visit_ratio"],
    s=20 + 10 * df_corr["n_keywords"],
    cmap="viridis",
)
axes[1].set_title("corr_pre_post vs min_visits")
axes[1].set_xlabel("min_visits")
cbar2 = plt.colorbar(sc2, ax=axes[1])
cbar2.set_label("max_visit_ratio")

plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3 : Procrustes alignment  (align post-periods onto pre_covid)
# ═══════════════════════════════════════════════════════════════════════════════
SCHEME = 1
model_ref = models[f"pre_covid_scheme{SCHEME}"]

aligned_models = {f"pre_covid_scheme{SCHEME}": model_ref}
period_name = "post_immediate"
anchors_list1 = select_anchor_pois(
    df_dist,
    PERIODS,
    keywords=["school", "mall"],
    min_visits_per_period=20,
    max_visit_ratio=1.4,
)
aligned, R = place2vec.align_model_procrustes(
    model_source=model_ref,
    model_target=models[f"{period_name}_scheme{SCHEME}"],
    anchor_tokens=anchors_list1,  # auto-detect HOME_*/WORK_*
    min_anchor=10,
    scheme=SCHEME,
)
aligned.save(f"models/place2vec_{period_name}_scheme{SCHEME}_aligned.model")
aligned_models[f"{period_name}_scheme{SCHEME}"] = aligned
print(f"✓ Aligned SCHEME={SCHEME} PERIOD={period_name}")

period_name = "post_1year"
anchors_list2 = select_anchor_pois(
    df_dist,
    PERIODS,
    keywords=["school", "mall"],
    min_visits_per_period=20,
    max_visit_ratio=1.6,
)
aligned, R = place2vec.align_model_procrustes(
    model_source=model_ref,
    model_target=models[f"{period_name}_scheme{SCHEME}"],
    anchor_tokens=anchors_list2,  # auto-detect HOME_*/WORK_*
    min_anchor=10,
    scheme=SCHEME,
)
aligned.save(f"models/place2vec_{period_name}_scheme{SCHEME}_aligned.model")
aligned_models[f"{period_name}_scheme{SCHEME}"] = aligned

## Load results after restarting kernel

In [ ]:
models = dict()
from gensim.models import KeyedVectors

for SCHEME in tqdm(range(4)):
    for period_name, month_list in PERIODS.items():
        model = KeyedVectors.load(
            f"models/place2vec_{period_name}_scheme{SCHEME}.model", mmap="r"
        )
        models[f"{period_name}_scheme{SCHEME}"] = model

In [ ]:
## FOR SCHEME 1-3
aligned_models = dict()
for SCHEME in tqdm(range(4)):
    model_ref = models[f"pre_covid_scheme{SCHEME}"]
    aligned_models[f"scheme{SCHEME}"] = {f"pre_covid": model_ref}
    for period_name in ["post_immediate", "post_1year"]:
        model = KeyedVectors.load(
            f"models/place2vec_{period_name}_scheme{SCHEME}_aligned.model", mmap="r"
        )
        aligned_models[f"scheme{SCHEME}"][f"{period_name}"] = model

## Step 4 Clustering

In [ ]:
# Elbow diagram: inertia vs k for each period and scheme
import place2vec

k_list = (2, 3, 4, 5, 6, 8, 10, 12)
df_elbow = place2vec.kmeans_elbow_scores(
    models,
    k_list=k_list,
    random_state=42,
    compute_silhouette=False,
)

fig = place2vec.plot_kmeans_elbow(
    df_elbow,
    periods=None,
    schemes=None,
    title_prefix="K-means elbow",
    figsize_per_subplot=(4, 3),
    sharey=False,
)
import matplotlib.pyplot as plt

if fig is not None:
    plt.show()

In [ ]:
import place2vec
import pandas as pd

dataset = "phd_datasets"
table_name = "place2vec_kmeans_clusters"
project = "phd-habidatum"
random_state = 42

dfs = []
for scheme in range(5):
    for period in PERIODS:
        if scheme == 0:
            k = 4
        else:
            k = 6
        if_table = "replace" if not dfs else "append"
        df_one = place2vec.kmeans_embeddings_to_bq(
            models[f"{period}_scheme{scheme}"],
            period=period,
            scheme=scheme,
            k=k,
            dataset=dataset,
            table_name=table_name,
            project=project,
            if_table_exist=if_table,
            random_state=random_state,
        )
        if not df_one.empty:
            dfs.append(df_one)

df_clusters = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

In [ ]:
df_clusters["poi_category"] = df_clusters["token"].apply(
    lambda x: (
        x.split("_")[0]
        if x.startswith("HOME") or x.startswith("WORK")
        else "_".join(x.split("_WEEKDAY")[0].split("_")[1:])
    )
)

In [ ]:
df_clusters["neighborhood_type"] = df_clusters["token"].apply(
    lambda x: (
        "residential remote"
        if "residential_remote" in x
        else (
            "satellite"
            if "satellite" in x
            else (
                "CBD"
                if "CBD" in x
                else "residential central" if "residential_central" in x else None
            )
        )
    )
)

In [ ]:
df_clusters["day_period"] = df_clusters["token"].apply(
    lambda x: x.split("WEEKDAY_")[-1] if "WEEKDAY_" in x else None
)

In [ ]:
df_clusters["token_type"] = df_clusters["token"].apply(
    lambda x: (
        "HOME" if x.startswith("HOME") else ("WORK" if x.startswith("WORK") else "POI")
    )
)

In [ ]:
df_clusters.to_csv("df_clusters_w2v.csv")

In [ ]:
# 1. Check what CAT_ tokens actually exist in the model
model_ref = models[f"post_1year_scheme3"]
cat_tokens = sorted(t for t in model_ref.wv.key_to_index if t.startswith("POI_"))

In [ ]:
## TODO: fails because doesnt have enough local memory
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4 : Save top 100 similar locations per token to BQ phd_datasets
# ═══════════════════════════════════════════════════════════════════════════════

for SCHEME in tqdm(range(4)):
    for i, (period_name, mdl) in enumerate(aligned_models.items()):
        place2vec.save_location_similarities(
            model=mdl,
            table_name=f"place2vec_location_similarities_period{period_name}_scheme{SCHEME}",
            dataset="phd_datasets",
            period_name=period_name,
            top_k=100,
            token_prefix=None,  # all tokens; use "POI_" for POIs only
            if_table_exist="replace",
        )

## Find leisure centroid vector

In [ ]:
# All tokens that contain "POI_bar"
leisure_tokens = [t for t in models['post_1year_scheme3'].wv.key_to_index if ('_bar') and ('EVENING' in t)]

In [ ]:
from place2vec import build_third_place_centroid
centroid_leisure, used = build_third_place_centroid(models["post_1year_scheme3"], leisure_tokens)

In [ ]:
import numpy as np

def avg_similarity_to_centroid(model, tokens, centroid):
    wv = model.wv
    present = [t for t in tokens if t in wv.key_to_index]
    if not present:
        return np.nan, []
    # cosine_similarities(one_vector, list_of_vectors) → array of similarities
    sims = wv.cosine_similarities(centroid, [wv[t] for t in present])
    return float(np.mean(sims)), np.min(sims), np.median(sims)

In [ ]:
from place2vec import avg_similarity_to_centroid

# Exclude negative correlations (default)
avg_sim, leisure_tokens_used = avg_similarity_to_centroid(
    models["post_1year_scheme3"], leisure_tokens, centroid_leisure, min_similarity=0.8
)

In [ ]:
centroid_leisure, used = build_third_place_centroid(models["post_1year_scheme3"], leisure_tokens_used)

### WORK CENTROID


In [ ]:
work_tokens = [t for t in models['post_1year_scheme3'].wv.key_to_index if 'WORK' in t]

In [ ]:
centroid_work, used = build_third_place_centroid(models["post_1year_scheme3"], work_tokens)

In [ ]:
from place2vec import avg_similarity_to_centroid

# Exclude negative correlations (default)
avg_sim, work_tokens_used = avg_similarity_to_centroid(
    models["post_1year_scheme3"], work_tokens, centroid_work, min_similarity=0.8
)

In [ ]:
from numpy.linalg import norm

def cosine_similarity(a, b):
    na, nb = norm(a), norm(b)
    if na == 0 or nb == 0:
        return np.nan
    return float(np.dot(a, b) / (na * nb))

similarity = cosine_similarity(centroid_work, centroid_leisure)

### Alternative way to find leisure places

In [ ]:
from place2vec import build_work_centroid

model = models["post_1year_scheme3"]  # or whichever period
wv = model.wv

# All tokens, ordered by similarity to WORK (highest first)
most_similar_to_work = wv.similar_by_vector(centroid_work, topn=len(wv))

# Most remote = lowest similarity = reverse order
most_remote_from_work = most_similar_to_work[::-1]

# Top 20 most remote
for token, sim in most_remote_from_work[:20]:
    print(f"{token}: {sim:.4f}")

In [ ]:
far_from_work_tokens = [token for token, sim in most_remote_from_work[:100] if 'EVENING' in token or 'NIGHT' in token]

In [ ]:
centroid_far_from_work, used = build_third_place_centroid(models["post_1year_scheme3"], far_from_work_tokens)

In [ ]:
# Exclude negative correlations (default)
avg_sim, far_from_work_tokens_used = avg_similarity_to_centroid(
    models["post_1year_scheme3"], far_from_work_tokens, centroid_far_from_work, min_similarity=0.3
)

In [ ]:
centroid_far_from_work, used = build_third_place_centroid(models["post_1year_scheme3"], far_from_work_tokens_used )


In [ ]:
from place2vec import (
    build_work_centroid,
    build_third_place_centroid,
    poi_work_thirdplace_similarities,
)

df = poi_work_thirdplace_similarities(model, centroid_work, centroid_far_from_work)

# POIs that look like work in the morning and third place later
flagged_before = df[df["work_morning_heavy_and_thirdplace_later"]]

In [ ]:
from place2vec import build_work_centroid, pois_high_work_similarity_daytime

model = models["post_1year_scheme3"]

# POIs with WORK similarity > 0.8 in morning/late_morning/afternoon, with category
df = pois_high_work_similarity_daytime(model, centroid_work, threshold=0.8)

In [ ]:
rw_keywords =  ["bakery",
    "cafe",
    "cafeteria",
    "coffee_shop",
    "tea_house",
    "dessert_restaurant",
    "cat_cafe",
    "dog_cafe",
    "breakfast_restaurant",
    "brunch_restaurant",
    "library",
    "book_store",
    "cultural_center",
    "community_center"]

In [ ]:
df_rw_places = df[df.keyword.isin(rw_keywords)&(df['qualifies_remote_work'])&((df['Evening_WORK'].isnull())|(df['Evening_WORK']<0.6))]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.set_theme(style="white")
ax = sns.barplot(
    y=df_rw_places.keyword.value_counts().index,
    x=df_rw_places.keyword.value_counts().values,
    palette="muted"
)
ax.set_xlabel("Number of Remote Work Places")
ax.set_ylabel("Place Keyword")
ax.set_title("Remote Work Qualified POIs by Keyword")
ax.grid(False)  # Remove grid
for i, v in enumerate(df_rw_places.keyword.value_counts().values):
    ax.text(v + max(df_rw_places.keyword.value_counts().values)*0.01, i, str(v), va='center')
plt.tight_layout()
plt.show()

In [ ]:
import sys

project_root = "../.."
if project_root not in sys.path:
    sys.path.append(project_root)

from geofunctions import utils

In [ ]:
#load full dataset
utils.upload_table_to_bq(df,table_name = "third_places_word2vec", dataset = "phd_pois",
if_table_exist="replace")

#### TODO: Add filter for residential or central

In [ ]:
cafe_tokens = [t for t in models['pre_covid_scheme3'].wv.key_to_index if ('cafe' in t or 'coffee' in t)and 'LATE_MORNING' in t]
cafe_tokens_after = [t for t in models['post_1year_scheme3'].wv.key_to_index if ('cafe' in t or 'coffee' in t)and 'LATE_MORNING' in t]

In [ ]:
cafe_tokens_fin = [i for i in cafe_tokens_after if i in cafe_tokens]

In [ ]:
centroid_cafe, used = build_third_place_centroid(models["pre_covid_scheme3"], cafe_tokens_fin)
avg_similarity_to_centroid(models["pre_covid_scheme3"], cafe_tokens_fin, centroid_cafe)

In [ ]:
centroid_cafe, used = build_third_place_centroid(models["post_1year_scheme3"], cafe_tokens_fin)
avg_similarity_to_centroid(models["post_1year_scheme3"], cafe_tokens_fin, centroid_cafe)

### 1st closeness to work locations

In [ ]:
# Q2 – Workiness score
SCHEME = 1
df_work_dict = {}
for period_name, mdl in aligned_models.items():
    df_work = place2vec.compute_closeness(
        mdl,
        target="work",
        aggregate_by_category=False,
        allowed_day_parts=["MORNING", "LATE_MORNING", "AFTERNOON"],
        include_geohash_tokens=False,
        aggregate_by_context=False,
    )
    print(f"\nWorkiness ({period_name}) – top 20:")
    print(df_work.sort_values("workiness", ascending=False).head(20))
    df_work_dict[period_name] = df_work

In [ ]:
df_pre = pd.DataFrame(df_work_dict["pre_covid_scheme1"])
df_pre["period"] = "pre_covid"
df_post_immediate = pd.DataFrame(df_work_dict["post_immediate_scheme1"])
df_post_immediate["period"] = "post_immediate"
df_post_1year = pd.DataFrame(df_work_dict["post_1year_scheme1"])
df_post_1year["period"] = "post_1year"
df_all = pd.concat([df_pre, df_post_immediate, df_post_1year])

In [ ]:
df_all.to_csv("df_allpois_workiness_scheme1.csv")

In [ ]:
# Q2 – Workiness score
SCHEME = 1
df_home_dict = {}
for period_name, mdl in aligned_models.items():
    df_home = place2vec.compute_closeness(
        mdl,
        target="home",
        aggregate_by_category=False,
        allowed_day_parts=["MORNING", "LATE_MORNING", "AFTERNOON"],
        include_geohash_tokens=False,
        aggregate_by_context=False,
    )
    print(f"\nWorkiness ({period_name}) – top 20:")
    print(df_home.sort_values("workiness", ascending=False).head(20))
    df_home_dict[period_name] = df_home

In [ ]:
df_pre = pd.DataFrame(df_home_dict["pre_covid_scheme1"])
df_pre["period"] = "pre_covid"
df_post_immediate = pd.DataFrame(df_home_dict["post_immediate_scheme1"])
df_post_immediate["period"] = "post_immediate"
df_post_1year = pd.DataFrame(df_home_dict["post_1year_scheme1"])
df_post_1year["period"] = "post_1year"
df_all = pd.concat([df_pre, df_post_immediate, df_post_1year])

In [ ]:
df_all.to_csv("df_allpois_homeness_scheme1.csv")

TODO: see clouds of close dots

In [ ]:
# Q3 – Neighbourhood prediction for a specific place
neighbours = place2vec.get_neighbours(
    aligned_models["pre_covid"],
    anchor_token="POI_some_cafe_id",  # <-- replace
    top_k=10,
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5 : Analysis helpers  (Q1–Q4)
# ═══════════════════════════════════════════════════════════════════════════════
from tqdm import tqdm

top_ngrams_dict = {}
# Q1 – Top N-grams
for SCHEME in tqdm(range(4)):
    print(f"Scheme {SCHEME}")
    top_ngrams_dict[f"Scheme{SCHEME}"] = {}
    for period_name in PERIODS:
        corpus_text_uri = (
            f"gs://{GS_BUCKET}/{GS_PREFIX}/corpus_{period_name}_scheme{SCHEME}.txt"
        )
        line_corpus = place2vec.GCSLineSentences(corpus_text_uri)
        top_ngrams = place2vec.extract_top_ngrams(
            line_corpus,
            n=3,
            top_k=10,
            aggregate_by_category=True,
            aggregate_by_neighborhood_type=True,
            aggregate_by_context=False,
            split_by_home_neighborhood=True,
        )
        top_ngrams_dict[f"Scheme{SCHEME}"][period_name] = top_ngrams
        print(f"\n=== {period_name} — Top 20 trigrams ===")
        print(top_ngrams)

In [ ]:
for scheme in ["Scheme1", "Scheme2", "Scheme3"]:
    print(scheme)
    for n in ["satellite_city", "residential_remote", "CBD", "residential_central"]:
        print(n)
        print("pre_covid")
        print([top_ngrams_dict[scheme]["pre_covid"][n]])
        print("post_immediate")
        print(top_ngrams_dict[scheme]["post_immediate"][n])
        print("post_1year")
        print(top_ngrams_dict[scheme]["post_1year"][n])
        print("-" * 100)
    print("-" * 100)

In [ ]:
# Q2 – Workiness score
for period_name, mdl in aligned_models.items():
    df_work = place2vec.compute_workiness(mdl)
    print(f"\nWorkiness ({period_name}) – top 10:")
    print(df_work.sort_values("workiness", ascending=False).head(10))

# Q3 – Neighbourhood prediction for a specific place
neighbours = place2vec.get_neighbours(
    aligned_models["pre_covid"],
    anchor_token="POI_some_cafe_id",  # <-- replace
    top_k=10,
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5 : Validation
# ═══════════════════════════════════════════════════════════════════════════════

# 4a. Stability check – supply a dict mapping category → list of POI tokens
category_map = {
    "gas_station": ["POI_gas1", "POI_gas2"],  # <-- replace with real IDs
    "supermarket": ["POI_super1", "POI_super2"],
}

for period_name, mdl in aligned_models.items():
    df_stab = place2vec.validate_stability(mdl, category_map, top_k=5)
    print(f"Stability ({period_name}):")
    print(df_stab.groupby("category")["matching_rate"].mean())

# 4b. Remote-work rank test – supply ground-truth remote-work POI IDs
ground_truth_rw_pois = ["poi_id_1", "poi_id_2", "poi_id_3"]  # <-- replace

for period_name, mdl in aligned_models.items():
    df_rank = place2vec.validate_remote_work_rank(
        mdl,
        ground_truth_poi_ids=ground_truth_rw_pois,
    )
    print(f"RW rank ({period_name}): median = {df_rank['rank'].median()}")